# 5주차 — 프롬프트 심화와 LCEL (Colab판)

「최신인공지능」 2026 · 5주차 실습

| 실습 | 교시 | 내용 |
|------|------|------|
| — | 1교시 | Few-shot — 예시는 지시문이 아니라 **데이터** |
| 확인 A | 1교시 | "JSON으로 답해줘" 는 정말 JSON으로 오는가 |
| — | 2교시 | `MessagesPlaceholder` 와 `partial()` |
| 실습 1 ★ | 2교시 | `with_structured_output()` — 부탁을 계약으로 |
| 실습 2 ★★ | 2교시 | **실패율을 직접 센다 (A/B)** |
| — | 3교시 | LCEL 과 Runnable |
| 실습 3 ★★ | 3교시 | Least-to-Most — 쪼개고·풀고·합친다 |
| 실습 4 ★ | 3교시 | Self-Consistency — N회 샘플링 후 다수결 |

> ✅ **이 차시는 Colab 이 실습실보다 유리합니다.** 외부 서비스 의존이 없고,
> 실습 2·4는 같은 호출을 수십 번 반복하므로 GPU 런타임에서 훨씬 빠릅니다.
>
> ⚠️ 다만 **로컬 소형 모델(1b)에서는 `with_structured_output()` 이 자주 실패**합니다.
> **[런타임] > [런타임 유형 변경] > T4 GPU** 로 먼저 바꿔 4b 를 쓰십시오.

## 0. 환경 준비

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
WEEK_MODELS   = ["chat"]
WEEK_PACKAGES = "langchain langchain-core langchain-ollama python-dotenv pydantic"
WEEK_SECRETS  = []

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

GPU   = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT  = os.environ.setdefault("MODEL",       "gemma3:4b" if GPU else "gemma3:1b")
SMALL = os.environ.setdefault("SMALL_MODEL", "gemma3:1b")
EMBED = os.environ.setdefault("EMBED_MODEL", "nomic-embed-text")
TOOL  = os.environ.setdefault("TOOL_MODEL",  "qwen3:4b")
PICK  = {"chat": CHAT, "small": SMALL, "embed": EMBED, "tool": TOOL}

print(f"[1/5] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  대화 모델 {CHAT}")
if not GPU:
    print("       ⚠️ 이 차시는 구조화 출력이 주제입니다. 1b 로는 실패율이 지나치게 높습니다.")
    print("       [런타임] > [런타임 유형 변경] > T4 GPU 로 바꾸고 이 셀을 다시 실행하십시오.")

print("[2/5] 패키지 설치 중…")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

if shutil.which("ollama") is None:
    print("[3/5] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/5] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/5] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
for key in WEEK_MODELS:
    name = PICK[key]
    if name in have:
        print(f"[5/5] {name:<20s} ✅ 이미 있음")
        continue
    print(f"[5/5] {name:<20s} ⏳ 내려받는 중… (진행 표시 없이 수 분 걸립니다)")
    t0 = time.time()
    r = sh(f"ollama pull {name}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")
    if r.returncode != 0:
        print(r.stderr[-400:])

print("\n" + "=" * 62)
print(f"준비 완료 — MODEL='{CHAT}'")
print("=" * 62)

## 1교시 1-3절 — Few-shot: 예시는 지시문이 아니라 데이터다

작년 방식은 예시를 프롬프트 **문자열 안에** 직접 적어 넣었습니다.

```python
prompt = '''다음 문장의 감정을 분류하세요.

문장: 배송이 빨라서 좋았어요   → 긍정
문장: 화면에 흠집이 있네요     → 부정

문장: {text} →'''
```

동작은 합니다. 그런데 예시를 5개 → 20개로 늘리려면?

| 하려는 일 | 문자열에 박아 넣었을 때 |
|---|---|
| 예시 추가·삭제 | 프롬프트 본문을 매번 편집 — 오타·형식 깨짐 |
| 예시만 교체 | 지시문까지 통째로 복사 |
| 예시 개수 실험 | 3개/5개/10개 버전을 파일 3개로 관리 |
| 예시를 DB·CSV에서 읽기 | ❌ **불가능** ★ |

★ 오늘 방식: **예시를 리스트(데이터)로 분리**합니다.
그러면 `examples` 를 CSV·DB에서 읽어와도 나머지 코드는 그대로입니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# ── ① 예시는 데이터다 — CSV·DB에서 읽어와도 된다 ★ ──────────
EXAMPLES = [
    {"text": "배송이 빨라서 좋았어요", "label": "긍정"},
    {"text": "화면에 흠집이 있네요",   "label": "부정"},
    {"text": "가격은 적당합니다",      "label": "중립"},
]


def build_prompt(examples: list[dict]) -> ChatPromptTemplate:
    # ── ② 예시 하나를 어떤 모양의 '대화'로 펼칠지 정한다 ────
    example_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "{text}"),
            ("ai",    "{label}"),
        ]
    )

    # ── ③ 둘을 합치면 Few-shot 블록이 된다 ──────────────────
    few_shot = FewShotChatMessagePromptTemplate(
        examples=examples,
        example_prompt=example_prompt,
    )

    return ChatPromptTemplate.from_messages(
        [
            ("system", "문장의 감정을 긍정/부정/중립 중 하나로만 답하세요."),
            few_shot,                    # ← 예시 블록이 통째로 끼워진다 ★
            ("human", "{text}"),
        ]
    )


prompt = build_prompt(EXAMPLES)
messages = prompt.invoke({"text": "포장이 엉망이었어요"}).to_messages()

print("── 모델에게 실제로 전달되는 메시지 ─────────────────")
for m in messages:
    print(f"  {type(m).__name__:15s} {m.content}")

# ★ 관찰 포인트
#   예시가 ("human", ...) / ("ai", ...) 대화 쌍으로 들어간다.
#   지시문 안의 텍스트가 아니라 "이미 이렇게 대화한 적이 있다" 는 형태다.
#   채팅 모델에서는 이쪽이 대체로 더 잘 먹힌다.

In [ ]:
# ── ④ 예시 개수만 바꿔 실험한다 — 코드는 그대로 ★ ──────
print("── 예시 개수를 바꿔도 코드는 그대로 ────────────────")
for k in (1, 2, 3):
    n_msgs = len(build_prompt(EXAMPLES[:k]).invoke({"text": "..."}).to_messages())
    print(f"  예시 {k}개 → 전달 메시지 {n_msgs}개")

print()
print("  문자열에 박아 넣었다면 이 실험을 하려고 프롬프트를 3벌 만들어야 했다.")
print()
print("💡 입력에 따라 예시를 골라 넣는 것(SemanticSimilarityExampleSelector)도 가능하다.")
print("   다만 임베딩 유사도(2주차) + 벡터 저장소(11주차)가 필요하다 → 11주차에 재료가 갖춰진다.")

## 1교시 1-5절 확인 A — "JSON으로 답해줘" 는 정말 JSON으로 오는가

프롬프트로 형식을 **'부탁'** 했을 때 무엇이 오는지 눈으로 봅니다.
10번 돌리면 아래 다섯 가지가 섞여 나옵니다.

| 유형 | 예 | 언제 터지나 |
|---|---|---|
| ① 정상 | `{"product": ..., "rating": 4}` | — |
| ② 앞말 붙임 | `물론이죠! {...}` | `json.loads()` 즉시 |
| ③ 코드펜스 | ` ```json ... ``` ` | `json.loads()` 즉시 |
| ④ 타입 틀림 ⚠️ | `"rating": "4점"` | **한참 뒤** ★ |
| ⑤ 필드명 틀림 ⚠️ | `"제품": ...` | **한참 뒤** ★ |

> ②③은 `json.loads()` 에서 즉시 터집니다 — 바로 압니다.
> **④⑤는 파싱이 '성공'합니다.** 그리고 한참 뒤 `data["rating"] + 1` 같은 줄에서 터집니다.
> → **조용한 실패.** 원인 추적이 어렵습니다.
>
> ★ `temperature` 가 0이 아닙니다. 0이면 10번 다 같은 답이 나와 흔들림이 안 보입니다.

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL  = os.environ["MODEL"]
TEMP   = 0.7      # ★ 0이 아님 — 형식이 흔들리는 장면을 봐야 한다
N      = 10
REVIEW = "이 무선 이어폰 배터리는 정말 오래갑니다. 다만 케이스가 좀 크네요."

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 리뷰 분석기다. 반드시 JSON으로만 답하라. 설명을 덧붙이지 마라."),
        ("human",  "다음 리뷰를 분석해라. 필드는 product, rating(1~5 정수), summary 다.\n\n{review}"),
    ]
)
chain = prompt | ChatOllama(model=MODEL, temperature=TEMP) | StrOutputParser()

for i in range(N):
    print(f"--- {i + 1}회 " + "-" * 45)
    print(chain.invoke({"review": REVIEW})[:200])

> 프롬프트를 더 강하게 쓰면(반드시·절대·오직 JSON만) 실패율은 **조금** 줄어듭니다.
> 그러나 **0이 되지는 않습니다.** 부탁은 부탁이기 때문입니다.
>
> ### 📌 부탁은 확률이고, 스키마는 계약이다.
>
> → 2교시 실습 1·2에서 숫자로 확인합니다.

## 2교시 1절 — `MessagesPlaceholder` 와 `partial()`

1교시에서 프롬프트를 고정 부분과 가변 부분으로 나눴습니다.
그런데 **'개수를 모르는 것'** 이 하나 있습니다 — 대화 이력입니다.

```
1턴째:  system + human
2턴째:  system + human + ai + human
3턴째:  system + human + ai + human + ai + human
                 └──────── 늘어난다 ────────┘
```

| 자리 | 받는 것 |
|---|---|
| `{question}` | 문자열 하나 |
| `MessagesPlaceholder("history")` | 메시지 **여러 개** ★ |

★ 오늘은 **자리를 비워 두는 것까지만** 합니다.
이 자리를 누가 채워 주느냐 = 메모리 → **11주차(대화형 RAG) · 13주차(에이전트)**

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 상담원입니다."),
        MessagesPlaceholder("history"),      # ← 메시지 '여러 개' 가 들어갈 자리 ★
        ("human", "{question}"),
    ]
)

print("── 이력이 있을 때 ──────────────────────────")
messages = prompt.invoke(
    {
        "history": [                          # ← 리스트를 넘긴다
            HumanMessage("환불 규정이 어떻게 되나요?"),
            AIMessage("구매 후 7일 이내 가능합니다."),
        ],
        "question": "영수증이 없어도 되나요?",
    }
)
for m in messages.to_messages():
    print(f"  {type(m).__name__:15s} {m.content}")

print()
print("── 이력이 아직 없을 때 (빈 리스트) ─────────")
empty = prompt.invoke({"history": [], "question": "영업시간이 어떻게 되나요?"})
for m in empty.to_messages():
    print(f"  {type(m).__name__:15s} {m.content}")

In [ ]:
from datetime import date
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {role} 전문가입니다. {audience} 눈높이로 답하세요."),
        ("human",  "{question}"),
    ]
)

print("── partial 없이 — 매번 3개를 다 넘겨야 한다 ─")
print("  ", prompt.input_variables)

print()
print("── partial 로 미리 고정 ★ ─────────────────")
py_tutor = prompt.partial(role="파이썬", audience="초보자")
print("  ", py_tutor.input_variables)         # ['question'] ← 이제 하나만 넘기면 된다

for q in ("리스트란?", "튜플이란?"):
    system = py_tutor.invoke({"question": q}).to_messages()[0]
    print(f"   {q:10s} → system: {system.content}")

# ── 쓰임새 ① 역할 특화 템플릿 파생 ─────────────
print()
print("── 하나의 템플릿에서 여러 파생본 ──────────")
for name, role in [("py_tutor", "파이썬"), ("db_tutor", "데이터베이스"), ("net_tutor", "네트워크")]:
    derived = prompt.partial(role=role, audience="초보자")
    print(f"   {name:10s} {derived.invoke({'question': '...'}).to_messages()[0].content}")

In [ ]:
# ── 쓰임새 ② 호출 시점에 계산되는 값 ★ ─────────
from datetime import date
from langchain_core.prompts import ChatPromptTemplate

dated_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {role} 전문가입니다. 오늘은 {today} 입니다."),
        ("human",  "{question}"),
    ]
)

# ⚠️ partial(today=date.today()) 처럼 '값' 을 넘기면
#    프로그램이 켜진 순간의 날짜로 굳는다. 서버가 며칠 떠 있으면 날짜가 안 바뀐다.
frozen = dated_prompt.partial(role="파이썬", today=date.today().isoformat())

# ✅ '함수' 를 넘기면 매 호출마다 계산된다.
live = dated_prompt.partial(role="파이썬", today=lambda: date.today().isoformat())

print("── 값이 아니라 '함수' 를 넘긴다 ★ ─────────")
for label, p in [("굳음 (값)", frozen), ("살아있음 (함수) ★", live)]:
    print(f"   {label:20s} {p.invoke({'question': '...'}).to_messages()[0].content}")

print()
print("  📌 partial 이라는 이름은 functools.partial 과 같은 발상이다.")
print("     인자 일부를 미리 묶어 새 함수를 만드는 것 — 여기서는 새 '프롬프트' 를 만든다.")

## 실습 1 ★ (2교시) — `with_structured_output()`: 부탁을 계약으로

확인 A에서는 프롬프트로 **'부탁'** 했습니다. 여기서는 스키마로 **'강제'** 합니다.
결과가 문자열이 아니라 `Review` **객체**로 옵니다.

```
[3주차~4주차]  prompt | llm | StrOutputParser()            → str
[5주차 오늘]   prompt | llm.with_structured_output(Review) → Review 객체 ★
                                ▲
                         parser 자리가 스키마로 '승격' 되었다
```

★ 4주차에는 **model 자리**를 갈아끼웠습니다. 오늘은 **parser 자리**를 갈아끼웁니다.
체인의 나머지는 그대로 — 이것이 Runnable 규약의 이득입니다.

> ⚠️ `with_structured_output()` 을 쓰면 **파서를 따로 붙이지 않습니다.**
> 뒤에 `| StrOutputParser()` 를 또 붙이면 객체가 다시 문자열로 뭉개집니다.

In [ ]:
from pydantic import BaseModel, Field, ValidationError


# ── ① 계약서를 쓴다 ────────────────────────────────
class Review(BaseModel):
    """고객 리뷰에서 뽑아낸 분석 결과."""      # ← 이 docstring 도 모델에게 전달된다 ★

    product: str       = Field(description="리뷰 대상 제품명")
    rating:  int       = Field(description="1~5 사이 정수 별점", ge=1, le=5)
    summary: str       = Field(description="30자 이내 한 줄 요약")
    pros:    list[str] = Field(description="장점 목록", default_factory=list)
    cons:    list[str] = Field(description="단점 목록", default_factory=list)


# ★★ Field(description=...) 은 주석이 아니다.
#    LangChain 이 이 스키마를 JSON Schema 로 변환해 모델에게 실제로 보낸다.
#    즉 description 을 잘 쓰는 것이 곧 프롬프트 엔지니어링이다.
#    1교시의 "명확·구체" 원칙이 여기로 옮겨온 것.

print("── Pydantic 검증은 이 자리에서 터진다 ──────")
cases = [
    ("정상",      dict(product="이어폰", rating=4,     summary="배터리 우수")),
    ("타입 틀림", dict(product="이어폰", rating="4점", summary="...")),
    ("범위 초과", dict(product="이어폰", rating=9,     summary="...")),
    ("필드 누락", dict(product="이어폰")),
]
for label, kwargs in cases:
    try:
        Review(**kwargs)
        print(f"  {label:10s} ✅ 통과")
    except ValidationError as e:
        print(f"  {label:10s} ❌ ValidationError ({e.error_count()}건)")

print()
print("  📌 실패가 뒤로 미뤄지지 않고 '여기서' 터진다.")
print("     1교시의 '한참 뒤에 터지는 조용한 실패' 가 호출 직후로 앞당겨졌다.")

In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL  = os.environ["MODEL"]
TEMP   = 0      # ★ 구조화 출력은 낮게 (Self-Consistency 는 반대 — 3교시)
REVIEW = "이 무선 이어폰 배터리는 정말 오래갑니다. 소리도 깨끗해요. 다만 케이스가 좀 크네요."

# ── ② 프롬프트에서 형식 지시를 뺀다 ★ ────────────
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 고객 리뷰 분석기다."),   # "JSON으로 답하라" 가 사라졌다 ★
        ("human",  "다음 리뷰를 분석해라.\n\n{review}"),
    ]
)

# ── ③ 모델에 스키마를 물린다 ─────────────────────
llm = ChatOllama(model=MODEL, temperature=TEMP)
chain = prompt | llm.with_structured_output(Review)

# 🔶 방식이 잘 안 맞으면 명시적으로 지정한다
#    chain = prompt | llm.with_structured_output(Review, method="json_schema")

result = chain.invoke({"review": REVIEW})

print("── 결과 ────────────────────────────────────")
print("  type      :", type(result))     # Review  ← 문자열이 아니다 ★
print("  product   :", result.product)
print("  rating    :", result.rating, "★")
print("  summary   :", result.summary)
print("  pros      :", result.pros)
print("  cons      :", result.cons)
print()
print("  rating + 1:", result.rating + 1, " ← int 라서 바로 계산된다 ★")
print("  model_dump:", result.model_dump())      # dict 로 변환 (저장·전송용)

### 관찰 — 무엇이 달라졌나

| 관찰 항목 | 짚어 볼 말 |
|---|---|
| 프롬프트에서 형식 지시가 사라짐 | 형식은 이제 프롬프트가 아니라 **스키마의 일** ★ |
| 반환값이 문자열이 아님 | `result["rating"]` 이 아니라 `result.rating` |
| `rating` 이 `int` | 바로 계산·정렬·평균에 쓸 수 있다 |
| 파서를 안 붙였음 | `with_structured_output` 이 파서 역할까지 한다 |
| 편집기 자동완성 | `result.` 을 치면 필드가 뜬다 |

> 💡 **한 걸음 더**: `Field(description=...)` 을 일부러 지우고 다시 돌려 보십시오.
> 품질이 떨어지는 것이 보이면 "**description 이 곧 프롬프트**" 가 몸으로 전달됩니다.
>
> ⚠️ 스키마를 붙였다고 모델이 실수를 안 하게 되는 것은 아닙니다.
> 실수를 **'즉시·확실히 잡아낸다'** 는 것이 이득입니다. → 실습 2에서 숫자로 확인

## 실습 2 ★★ (2교시) — 실패율을 직접 센다

핵심 질문: **"프롬프트를 더 강하게 쓰면 되지 않나요?"** → 숫자로 답합니다.

```
같은 리뷰 · 같은 모델 · 같은 온도로 각각 N회

방법 A: "반드시 JSON으로만 답하라"  ──▶ json.loads() + 필드/타입 검사
방법 B: with_structured_output()    ──▶ 스키마 검증

                  실패 횟수를 센다
```

★ **판정 기준을 A·B 양쪽에 똑같이 적용하는 것이 중요합니다.**
방법 A를 `json.loads()` 성공 여부로만 재면 A에 유리하게 왜곡됩니다.
그래서 A도 파싱 후 **같은 `Review` 스키마로 검증**합니다.

| 무엇을 실패로 세나 | |
|---|---|
| JSON 파싱 자체가 안 됨 | ❌ |
| 파싱은 됐는데 필드가 없음 | ❌ |
| 파싱은 됐는데 타입이 다름 (`"4점"`) | ❌ ★ |
| 값이 범위를 벗어남 (`rating=9`) | ❌ |

> ⚠️ `temperature` 를 A·B에 **똑같이** 적용하십시오. 한쪽만 0으로 두면 실험이 성립하지 않습니다.
> 학생이 "B가 이긴 건 온도를 낮춰서 아니냐"고 물으면 — **그 질문이 나오면 좋은 수업입니다.**
>
> 🔶 **시간 관리**: 20회 × 2 = 40회면 수 분이 걸립니다. 실행을 걸어 둔 뒤 해석을 진행하고,
> 끝난 순서대로 결과를 받으십시오. 빠듯하면 `N = 10` 으로 낮춰도 경향은 보입니다.

In [ ]:
import json, os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field, ValidationError

MODEL  = os.environ["MODEL"]
N      = 20       # 시간이 빠듯하면 10 으로 낮추십시오
TEMP   = 0.7      # ★ 0이면 매번 같은 답 → 차이가 안 보인다
REVIEW = "이 무선 이어폰 배터리는 정말 오래갑니다. 다만 케이스가 좀 크네요."


class Review(BaseModel):
    """고객 리뷰 분석 결과."""

    product: str = Field(description="제품명")
    rating:  int = Field(description="1~5 정수 별점", ge=1, le=5)
    summary: str = Field(description="30자 이내 요약")


# ══════════════════════════════════════════════════
# 방법 A — 프롬프트로 부탁
# ══════════════════════════════════════════════════
prompt_a = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 리뷰 분석기다. 반드시 JSON으로만 답하라. 설명을 덧붙이지 마라."),
        ("human",  "다음 리뷰를 분석해라. 필드는 product(문자열), "
                   "rating(1~5 정수), summary(문자열) 이다.\n\n{review}"),
    ]
)
chain_a = prompt_a | ChatOllama(model=MODEL, temperature=TEMP) | StrOutputParser()

# ══════════════════════════════════════════════════
# 방법 B — 스키마로 강제
# ══════════════════════════════════════════════════
prompt_b = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 리뷰 분석기다."),
        ("human",  "다음 리뷰를 분석해라.\n\n{review}"),
    ]
)
chain_b = prompt_b | ChatOllama(model=MODEL, temperature=TEMP).with_structured_output(Review)


def run_a() -> str | None:
    """실패면 사유 문자열, 성공이면 None."""
    text = chain_a.invoke({"review": REVIEW})
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return "JSON 파싱 실패"          # 앞말 · 코드펜스

    if not isinstance(data, dict):
        return "JSON이 객체가 아님"

    try:
        Review(**data)                    # ★ B와 같은 잣대로 검증한다
    except ValidationError as e:
        return f"스키마 불일치({e.error_count()}건)"   # 타입 · 필드명 · 범위
    except TypeError:
        return "필드명 불일치"
    return None


def run_b() -> str | None:
    try:
        chain_b.invoke({"review": REVIEW})
    except Exception as e:
        return type(e).__name__
    return None


print(f"모델={MODEL}  temperature={TEMP}  N={N}   (A·B 동일 조건 ★)")
print("=" * 60)

summary = []
for label, fn in [("A  프롬프트로 부탁", run_a), ("B  스키마로 강제", run_b)]:
    fails = []
    print(f"{label} : ", end="", flush=True)
    for i in range(N):
        reason = fn()
        print("." if reason is None else "X", end="", flush=True)
        if reason:
            fails.append((i + 1, reason))
    rate = len(fails) / N * 100
    print(f"\n[{label}]  실패 {len(fails)}/{N}  =  {rate:.0f}%")
    for i, r in fails:
        print(f"    - {i}회차: {r}")
    print("-" * 60)
    summary.append((label, len(fails), rate))

print()
print("결과 기록표 — 저장소에 함께 커밋할 것 ★")
print("  방법                          실패 횟수      실패율")
print("  " + "─" * 52)
for label, n_fail, rate in summary:
    print(f"  {label:28s} {n_fail:>3d} / {N}      {rate:>5.0f} %")

### 읽어낼 것

| 관찰 | 의미 |
|---|---|
| A의 실패율 > B의 실패율 | **부탁은 확률, 스키마는 계약** ★ |
| A의 실패 사유가 제각각 | 앞말·코드펜스·타입·필드명 — 예외 처리를 몇 개나 짜야 하나 |
| B도 0%가 아닐 수 있음 ⚠️ | 스키마도 만능은 아니다 (아래) |
| B의 실패는 **즉시** 예외 | A의 실패는 한참 뒤에 터진다 |

### ⚖️ B가 0%가 아니어도 당황하지 마십시오 — 오히려 더 좋은 수업 재료입니다

- 필드 값이 말이 안 됨 (`rating=1` 인데 극찬 리뷰)
  → 스키마는 **'형식'** 을 보장하지, **'내용'** 을 보장하지 않는다 ★
- 모델이 스키마를 못 지킴
  → 소형 로컬 모델의 한계. `with_structured_output(..., method="json_schema")` 로 조정

내용의 진위 검증은 **7주차 평가(Evaluation)** 의 주제입니다.

### 📌 결론

> "프롬프트를 강하게 쓰는 것으로는 0%에 못 간다.
> **형식은 프롬프트가 아니라 타입 시스템이 지켜야 한다.**"

## 3교시 1절 — LCEL 과 Runnable: 파이프가 실제로 하는 일

3주차부터 계속 써 온 한 줄:

```python
chain = prompt | llm | parser
```

이 문법에 이름이 있습니다 — **LCEL** (LangChain Expression Language).
그리고 `|` 로 이어붙일 수 있는 것들의 공통 규약이 **Runnable** 입니다.

| 메서드 | 하는 일 |
|---|---|
| `.invoke(입력)` | 하나 실행 |
| `.batch([입력들])` | 여러 개 병렬 실행 ★ 실습 4 |
| `.stream(입력)` | 토큰 단위로 흘려보냄 (4주차) |
| `.ainvoke` / `.abatch` / `.astream` | 각각의 비동기판 |

★ **이 메서드들을 가진 것은 전부 `|` 로 이을 수 있습니다.**
프롬프트도, 모델도, 파서도, 체인 자체도, 심지어 평범한 파이썬 함수도.

4주차에서 `ChatOllama` ↔ `ChatOpenAI` 를 갈아끼울 수 있었던 이유,
2교시에서 parser 자리를 스키마로 바꿀 수 있었던 이유가 전부 **이 규약 하나**입니다.

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

llm = ChatOllama(model=os.environ["MODEL"], temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 한 문장으로만 답하는 비서다."),
        ("human",  "{topic} 을 한 문장으로 요약해줘."),
    ]
)
chain = prompt | llm | StrOutputParser()

print("── 체인의 타입 ─────────────────────────────")
print("  ", type(chain))                     # RunnableSequence
print("  체인도 .invoke 를 가지므로, 다른 체인의 부품이 된다 ★")

#    prompt ──▶ llm ──▶ parser
#    └──────── chain ────────┘     ← 이 덩어리 자체가 다시 하나의 부품
#
# 📌 이 성질이 실습 3(Least-to-Most)의 전부다.
#    큰 체인을 만든다는 것은 작은 체인을 부품으로 쓰는 것이다.

print()
print("  invoke :", chain.invoke({"topic": "LCEL"})[:80])

In [ ]:
# ══════════════════════════════════════════════════
# RunnableLambda — 평범한 함수를 부품으로
# ══════════════════════════════════════════════════
from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_template("{topic} 의 장점을 3가지 알려줘.")


def shorten(text: str) -> str:
    return text[:100] + " ..."


chain = prompt | llm | StrOutputParser() | RunnableLambda(shorten)

print("── RunnableLambda — 함수도 부품이다 ────────")
print("  ", chain.invoke({"topic": "LangChain"}))
print()
print("  ⚠️ 함수는 인자를 하나만 받아야 한다. 여러 값을 넘기려면 dict 하나로 묶는다.")
print("     실습 3에서 이 방식으로 '반복문' 을 체인 안에 넣는다.")

In [ ]:
# ══════════════════════════════════════════════════
# RunnablePassthrough.assign — 입력을 보존하며 항목 추가 ★★
# ══════════════════════════════════════════════════
from langchain_core.runnables import RunnablePassthrough

keyword_chain = (
    ChatPromptTemplate.from_template("{text} 에서 핵심 키워드 3개만 쉼표로 나열해라.")
    | llm
    | StrOutputParser()
)

print("── assign 없이 — 원본이 사라진다 ───────────")
lost = keyword_chain.invoke({"text": "LCEL 은 Runnable 을 파이프로 잇는 문법이다."})
print("  ", repr(lost)[:90])
print("   ↑ 원래 입력 text 가 어디에도 없다. 뒷단계에서 또 써야 하는데 이미 없다.")

print()
print("── assign 으로 — 원본을 보존하며 옆에 붙인다 ★")
step = RunnablePassthrough.assign(keywords=keyword_chain)
kept = step.invoke({"text": "LCEL 은 Runnable 을 파이프로 잇는 문법이다."})
for k, v in kept.items():
    print(f"   {k:10s} {str(v)[:60]}")

#  입력  {"text": "..."}
#    ▼
#  출력  {"text": "...", "keywords": <결과>}     ← 원본이 살아 있다 ★
#
# ★ 10주차 RAG 에서 다시 나온다. 실습 3에서 바로 쓴다.

In [ ]:
# ══════════════════════════════════════════════════
# RunnableParallel — 두 갈래를 동시에
# ══════════════════════════════════════════════════
from langchain_core.runnables import RunnableParallel

summary_chain = (
    ChatPromptTemplate.from_template("{text} 를 한 문장으로 요약해라.") | llm | StrOutputParser()
)
keyword_chain = (
    ChatPromptTemplate.from_template("{text} 에서 키워드 3개만 쉼표로 나열해라.")
    | llm
    | StrOutputParser()
)

both = RunnableParallel(summary=summary_chain, keywords=keyword_chain)
# dict 리터럴로 써도 같다 (LCEL이 자동 변환)
#   both = {"summary": summary_chain, "keywords": keyword_chain}

print("── RunnableParallel — 다른 체인들을 동시에 ─")
result = both.invoke({"text": "Least-to-Most 는 큰 문제를 하위 질문으로 쪼개 순서대로 푼다."})
for k, v in result.items():
    print(f"   {k:10s} {str(v)[:70]}")

print()
print("  💡 batch() 와 다르다.")
print("     RunnableParallel = '다른 체인들' 을 같은 입력으로 동시에")
print("     batch()          = '같은 체인' 에 다른 입력들을 동시에   ← 실습 4는 이쪽 ★")

## 실습 3 ★★ (3교시) — Least-to-Most: 쪼개고 · 순서대로 풀고 · 합친다

**❌ 한 번에:**
> "이 제품 리뷰 50건을 읽고, 반복되는 불만을 찾고, 그 불만의 원인을 추정하고,
> 개선 우선순위를 정해서 보고서를 써라."

→ 앞의 지시를 하다가 뒤를 잊는다 / 근거 없이 결론부터 쓴다 / 중간을 검증할 수 없다

**✅ 쪼개서 (Least-to-Most):**

```
① 반복되는 불만은 무엇인가?          ← 쉽다. 사실 추출
     │  (①의 답을 근거로)
② 각 불만의 원인은?                  ← ①이 있어야 답할 수 있다
     │  (①②의 답을 근거로)
③ 개선 우선순위는?                   ← ①②가 있어야 답할 수 있다
     ▼
④ 종합 보고서
```

- **얻는 것**: 정확도(앞 결과가 근거) / 검증 가능성(어디서 틀렸는지 보임) / 재사용
- ⚠️ **대가도 있다**: 호출이 1회 → N+2회. 시간과 비용이 배로 든다.
  "쪼갤수록 좋다" 가 아니라 **"한 번에 안 되는 문제만 쪼갠다"**.

> ★ 이 체인이 **6주차 [과제 2]** 의 대상입니다.
> 단계가 여러 개여서 LangSmith 추적 화면이 풍성하게 나옵니다. **반드시 커밋하십시오.**

In [ ]:
import os
from typing import List
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

MODEL = os.environ["MODEL"]
TEMP  = 0        # ★ 분해·풀이는 낮은 온도 (Self-Consistency 는 반대)

PROBLEM = (
    "한 카페가 오후 시간대 매출만 계속 줄고 있다. "
    "원인을 진단하고 개선안을 우선순위와 함께 제시하라."
)

llm = ChatOllama(model=MODEL, temperature=TEMP)


# ── ① 분해기 — 하위 질문 목록을 '구조화 출력' 으로 받는다 ★ ─────
class SubQuestions(BaseModel):
    """원 문제를 풀기 위해 순서대로 답해야 할 하위 질문들."""

    questions: List[str] = Field(description="쉬운 것부터 어려운 순서로 3~4개")


decompose_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 문제를 잘게 쪼개는 조교다. 문제를 직접 풀지 마라."),
        ("human",  "다음 문제를 풀기 위해 순서대로 답해야 할 하위 질문으로 나눠라.\n\n{problem}"),
    ]
)
decomposer = decompose_prompt | llm.with_structured_output(SubQuestions)

# ⚠️ 분해 결과가 엉망일 때 고칠 곳은 코드가 아니라 SubQuestions 의 description 이다.
#    (2교시에서 말한 "description 이 곧 프롬프트")


# ── ② 풀이기 — 하위 질문 하나를 푼다 ────────────────────────────
solve_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 문제 해결 조교다. 앞서 푼 결과를 근거로 이번 질문에만 짧게 답하라."),
        ("human",  "원 문제: {problem}\n\n지금까지 푼 것:\n{solved}\n\n이번 질문: {question}"),
    ]
)
solver = solve_prompt | llm | StrOutputParser()


# ── ③ 순서대로 푸는 반복문을 '부품' 으로 만든다 ★ ───────────────
def solve_in_order(data: dict) -> dict:
    problem = data["problem"]
    solved: List[str] = []

    for i, q in enumerate(data["subs"].questions, 1):
        answer = solver.invoke(
            {
                "problem":  problem,
                "solved":   "\n\n".join(solved) or "(아직 없음)",   # ← 앞 결과를 넘긴다 ★
                "question": q,
            }
        )
        solved.append(f"Q{i}. {q}\nA{i}. {answer}")
        print("─" * 60)
        print(solved[-1])

    return {"problem": problem, "solved": "\n\n".join(solved)}


# ── ④ 종합기 ────────────────────────────────────────────────────
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 최종 답변자다. 하위 풀이만을 근거로 답하라."),
        ("human",  "원 문제: {problem}\n\n하위 풀이:\n{solved}\n\n최종 답을 정리해라."),
    ]
)


# ── ⑤ 전부 파이프로 잇는다 ★★ ──────────────────────────────────
chain = (
    RunnablePassthrough.assign(subs=decomposer)   # {"problem"} → {"problem", "subs"}
    | RunnableLambda(solve_in_order)              # → {"problem", "solved"}
    | final_prompt
    | llm
    | StrOutputParser()
)

#  {"problem"}
#       │
#       ├─ assign(subs=decomposer) ──▶ {"problem", "subs"}      ← 원본 보존 ★
#       │
#       ├─ RunnableLambda(solve_in_order) ──▶ {"problem", "solved"}
#       │        └ 내부에서 solver 를 하위 질문 수만큼 invoke (순차)
#       │
#       └─ final_prompt | llm | parser ──▶ 최종 답(str)

print("=" * 60)
print("원 문제:", PROBLEM)
print("=" * 60)

answer = chain.invoke({"problem": PROBLEM})

print("=" * 60)
print("[최종 답]")
print(answer)
print("=" * 60)

### 관찰 — 무엇을 볼 것인가

| 관찰 항목 | 짚어줄 말 |
|---|---|
| 하위 질문이 매번 조금씩 다름 | 분해도 모델이 하는 일 — 완전히 고정되지 않는다 |
| `solved` 가 누적됨 | 앞 답이 뒤 질문의 **'근거'** 로 들어가는 것이 핵심 ★ |
| 중간 출력이 화면에 보임 | 어디서 틀렸는지 추적 가능 → **다음 주 LangSmith 가 이걸 자동으로** |
| 호출 횟수 | 분해 1 + 하위 3~4 + 종합 1 = **5~6회** ⚠️ |
| `assign` 이 없으면 | `problem` 이 사라져 뒷단계가 깨진다 (일부러 지워 보여도 좋다) |

> 💡 **한 걸음 더**: 같은 문제를 '한 번에' 묻는 단일 프롬프트와 나란히 실행해 비교하십시오.
> "쪼갠 쪽이 근거가 있다" 가 눈으로 보이면 이 절이 완성됩니다.
>
> 📌 **이 체인이 6주차 [과제 2] 의 대상입니다. 반드시 커밋하십시오.**

## 실습 4 ★ (3교시) — Self-Consistency: N회 샘플링 후 다수결

```
같은 질문 ──┬──▶ CoT 1 ──▶ 17
            ├──▶ CoT 2 ──▶ 17          batch() 로 한 줄 ★
            ├──▶ CoT 3 ──▶ 23
            ├──▶ CoT 4 ──▶ 17
            └──▶ CoT 5 ──▶ 17
                           │
                     다수결 ▼
                          17
```

| 필요한 조건 | 이유 |
|---|---|
| `temperature > 0` ★ | 0이면 5번 다 같은 답 → 다수결이 무의미 |
| 답을 정확히 뽑아낼 수 있어야 | 문장에서 눈으로 찾으면 셀 수 없다 → **구조화 출력**(2교시) ★ |
| 답이 갈리는 문제 | 너무 쉬우면 5:0으로 끝나 아무것도 안 보인다 ⚠️ |

★ `batch()` 에 **'같은 입력을 N개'** 넣는 것이 이 실습의 요령입니다.

```python
[{"question": Q}] * N     # ← 이 한 줄이 Self-Consistency 의 구현이다
```

In [ ]:
import os
from collections import Counter
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

MODEL = os.environ["MODEL"]
N     = 5
TEMP  = 0.8      # ★ 0이면 다수결이 무의미하다 — 아래에서 0으로도 돌려 봅니다
MAX_CONCURRENCY = 2   # ⚠️ VRAM 보호 — 2~3으로 제한 🔶

# 🔶 답이 갈리는 문제 후보 — 수업 전에 각각 5회씩 돌려 보고 하나를 고를 것
QUESTIONS = [
    # ① 산술 추론 — 중간에 미끄러지기 쉬운 다단계 계산
    "한 상자에 사과가 12개씩 들어 있다. 상자 7개를 사서 그중 5개를 이웃에게 나눠 주고, "
    "남은 사과의 3분의 1을 잼으로 만들었다. 잼으로 만들지 않고 남은 사과는 몇 개인가?",
    # ② 논리 퍼즐 — 조건을 순서대로 반영해야 함
    "A는 B보다 나이가 많고, C는 A보다 어리지만 B보다는 많다. "
    "D는 셋 중 누구보다도 어리다. 나이가 많은 순서대로 나열하면?",
    # ③ 비율·속도 — 단위 처리에서 갈리기 쉬움
    "어떤 일을 혼자 하면 갑은 6시간, 을은 12시간이 걸린다. "
    "둘이 함께 2시간 일한 뒤 을이 빠지면, 갑이 혼자 마무리하는 데 몇 시간이 더 걸리는가?",
]
QUESTION = QUESTIONS[0]


class Solution(BaseModel):
    """단계별 추론과 최종 답."""

    reasoning: str = Field(description="단계별 풀이 과정")
    answer:    str = Field(description="최종 답만. 숫자면 단위 없이 숫자만")


# ★ answer 를 '단위 없이 숫자만' 으로 강제하는 것이 중요하다.
#   "17개" / "17 개" / "답은 17" 이 섞이면 Counter 가 다른 답으로 센다.

cot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 문제 해결가다. 단계적으로 생각한 뒤 최종 답을 낸다."),
        ("human",  "{question}"),
    ]
)


def self_consistency(temp: float) -> None:
    cot = cot_prompt | ChatOllama(model=MODEL, temperature=temp).with_structured_output(Solution)

    print("=" * 60)
    print(f"모델={MODEL}  temperature={temp}  N={N}")
    print("=" * 60)

    # ── ① 같은 입력을 N개 만들어 batch 로 넘긴다 ★ ──────────────
    results = cot.batch(
        [{"question": QUESTION}] * N,
        config={"max_concurrency": MAX_CONCURRENCY},   # 로컬 GPU 보호 ★
    )

    # ── ② 답만 뽑아 센다 ──────────────────────────────────────
    results = [r for r in results if r is not None]
    answers = [r.answer.strip() for r in results]

    for i, r in enumerate(results, 1):
        print(f"[{i}] 답={r.answer.strip():>10s}   근거={r.reasoning[:60]}...")

    votes = Counter(answers)
    final, count = votes.most_common(1)[0]

    print("-" * 60)
    print("표 분포 :", dict(votes))
    print(f"최종 답 : {final}   ({count}/{N} 표, {count / N * 100:.0f}%)")
    print(f"1회만 실행했다면 나왔을 답 : {answers[0]}   ← 운에 맡긴 결과")

    if len(votes) == 1:
        print("\n⚠️ 만장일치입니다. temperature 가 0이거나 문제가 너무 쉽습니다.")


print("문제:", QUESTION)
print()
self_consistency(TEMP)

In [ ]:
# ★ 먼저 temperature=0 으로도 돌려 보십시오 — 만장일치가 나옵니다.
#   샘플이 서로 달라야 다수결이 성립한다는 것을 눈으로 확인하는 셀입니다.
self_consistency(0.0)

### 읽어낼 것

| 관찰 | 의미 |
|---|---|
| `temp=0` → 만장일치 | 샘플이 서로 달라야 다수결이 성립한다 ★ |
| 소수 의견이 존재 | CoT 1회 실행은 그 소수 의견을 뽑을 수도 있었다 |
| 총 소요 시간 | 정확도를 **N배의 비용**으로 산 것 ⚠️ |
| 표가 3:2로 갈릴 때 | N을 늘려야 하는 신호 — 다만 비용도 함께 늘어난다 |

### ⚖️ 한계

| 해결하는 것 | 해결하지 못하는 것 |
|---|---|
| 추론 도중의 우연한 실수 | 모델이 **'일관되게' 틀리는** 경우 (만장일치 오답) ⚠️ |
| 1회 실행의 운 | 비용이 N배 — 모든 호출에 적용할 수 없다 |

> "다수결은 **자주 틀리는 것**을 걸러낼 뿐, **항상 틀리는 것**은 못 걸러낸다."
> 그건 7주차 평가(Evaluation)로 잡습니다.
>
> 📌 **적용 기준**: "틀리면 비용이 큰 소수의 판단" 에만 씁니다.
> 요약·번역에는 다수결을 쓰지 않습니다 — **셀 수 있는 답이 없기 때문**입니다. ★

## 오늘 확인할 것

- [ ] Few-shot 예시를 **데이터로 분리**해 개수를 바꿔 실험했다
- [ ] "JSON으로 답해줘" 가 흔들리는 것을 10회 실행으로 눈으로 봤다
- [ ] `with_structured_output()` 으로 **객체**를 받았다 ★
- [ ] A/B **실패율을 직접 세어** 기록표를 만들었다 ★★
- [ ] `assign` · `RunnableLambda` · `RunnableParallel` 을 각각 써 봤다
- [ ] **Least-to-Most 체인**을 완성했다 (6주차 과제 2 대상) ★★
- [ ] Self-Consistency 를 `temperature` 0 과 0.8 로 각각 돌려 비교했다 ★